# Milestone 3 - Context Augmentation with RAG Pipelines (WIP)

Exploring context retrieval to augment question prompts before option ranking.

In [ ]:
import os
import pandas as pd
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load train data
data_path = '../guides/train.csv' if os.path.exists('../guides/train.csv') else 'train.csv'
df = pd.read_csv(data_path)
print("Loaded train samples:", len(df))

# Initializing embedding model for vector index
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Embedding model ready.")

### Vector Indexing & Context Retrieval Function
Building a FAISS vector index over prompt texts and writing a helper function to retrieve top-k relevant contexts.

In [ ]:
# Encode corpus into FAISS index
corpus = df['prompt'].tolist()
embeddings = embedder.encode(corpus, show_progress_bar=False)
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))
print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}.")

# Context retrieval helper
def retrieve_context(query_str, top_k=2):
    q_emb = embedder.encode([query_str]).astype('float32')
    distances, indices = index.search(q_emb, top_k + 1)
    # Exclude self-match at index 0
    retrieved_texts = [corpus[idx] for idx in indices[0] if corpus[idx] != query_str][:top_k]
    return " ".join(retrieved_texts)

# Test retrieval on row 0
sample_query = df['prompt'].iloc[0]
sample_ctx = retrieve_context(sample_query, top_k=2)
print("Sample Query:", sample_query[:100], "...")
print("Retrieved Context:", sample_ctx[:100], "...")